# AeroPulse 3D - Environment Setup

Install only the packages not included in the base `jupyter/pyspark-notebook` image.

**If a cell times out, simply re-run it.** The base image already has PySpark, pandas, numpy, and pyarrow.

In [ ]:
# Check what's already installed in the base image
import sys
print(f"Python version: {sys.version}\n")

try:
    import pyspark
    print(f"✓ PySpark {pyspark.__version__} (pre-installed)")
except ImportError:
    print("✗ PySpark not found")

try:
    import pandas as pd
    print(f"✓ Pandas {pd.__version__} (pre-installed)")
except ImportError:
    print("✗ Pandas not found")

try:
    import numpy as np
    print(f"✓ NumPy {np.__version__} (pre-installed)")
except ImportError:
    print("✗ NumPy not found")

try:
    import pyarrow
    print(f"✓ PyArrow {pyarrow.__version__} (pre-installed)")
except ImportError:
    print("✗ PyArrow not found")

In [ ]:
# Install Delta Lake (not in base image)
!pip install --default-timeout=200 delta-spark==3.1.0

In [ ]:
# Install H3 spatial indexing
!pip install --default-timeout=200 h3==4.0.0b5

In [ ]:
# Install geospatial libraries (may take 2-3 minutes)
!pip install --default-timeout=200 geopandas==0.14.2 shapely==2.0.2 folium==0.15.1

In [ ]:
# Install visualization libraries
!pip install --default-timeout=200 pydeck==0.9.1 plotly==5.18.0

In [ ]:
# Install utilities
!pip install --default-timeout=200 tenacity==8.2.3 python-dotenv==1.0.0 pydantic==2.5.3

In [12]:
# Verify all required imports
print("Verifying installations...\n")

import pyspark
print(f"✓ PySpark {pyspark.__version__}")

import delta
print(f"✓ Delta Spark (installed)")

import h3
print(f"✓ H3 {h3.__version__}")

import pandas as pd
print(f"✓ Pandas {pd.__version__}")

import numpy as np
print(f"✓ NumPy {np.__version__}")

import pyarrow
print(f"✓ PyArrow {pyarrow.__version__}")

import geopandas as gpd
print(f"✓ GeoPandas {gpd.__version__}")

import pydeck as pdk
print(f"✓ PyDeck {pdk.__version__}")

import plotly
print(f"✓ Plotly {plotly.__version__}")

from tenacity import retry
print(f"✓ Tenacity (retry library)")

print("\n✅ All required dependencies installed!")

Verifying installations...

✓ PySpark 3.5.0
✓ Delta Spark (installed)
✓ H3 4.0.0b5
✓ Pandas 2.0.3
✓ NumPy 1.24.4
✓ PyArrow 13.0.0
✓ GeoPandas 0.14.2
✓ PyDeck 0.9.1
✓ Plotly 5.18.0
✓ Tenacity (retry library)

✅ All required dependencies installed!


In [13]:
# Initialize SparkSession with Delta Lake support
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("AeroPulse3D") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"✅ Spark {spark.version} with Delta Lake ready")
print(f"Spark UI: http://localhost:4040")

✅ Spark 3.5.0 with Delta Lake ready
Spark UI: http://localhost:4040


In [14]:
# Test H3 spatial indexing with North India locations
import h3

test_locations = [
    ("Delhi", 28.6139, 77.2090),
    ("Amritsar", 31.6340, 74.8723),
    ("Chandigarh", 30.7333, 76.7794),
]

print("H3 Resolution 7 indexing:\n")
for name, lat, lng in test_locations:
    h3_index = h3.latlng_to_cell(lat, lng, 7)
    print(f"{name:12} ({lat:.4f}, {lng:.4f}) → {h3_index}")

print("\n✅ H3 working correctly")

H3 Resolution 7 indexing:

Delhi        (28.6139, 77.2090) → 873da1146ffffff
Amritsar     (31.6340, 74.8723) → 87424d26dffffff
Chandigarh   (30.7333, 76.7794) → 873d14699ffffff

✅ H3 working correctly


In [15]:
# Create data directories
import os

dirs = [
    "/home/jovyan/work/data/raw",
    "/home/jovyan/work/data/delta",
    "/home/jovyan/work/data/checkpoints",
    "/home/jovyan/work/data/parquet"
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"✓ {d}")

print("\n✅ Directories ready")

✓ /home/jovyan/work/data/raw
✓ /home/jovyan/work/data/delta
✓ /home/jovyan/work/data/checkpoints
✓ /home/jovyan/work/data/parquet

✅ Directories ready


In [16]:
# Test Spark + H3 integration
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import h3

# H3 UDF
def lat_lng_to_h3(lat, lng, res=7):
    return h3.latlng_to_cell(lat, lng, res)

h3_udf = udf(lat_lng_to_h3, StringType())

# Test data
data = [
    ("Delhi", 28.6139, 77.2090, 150.5),
    ("Amritsar", 31.6340, 74.8723, 89.2),
    ("Chandigarh", 30.7333, 76.7794, 112.8),
]

df = spark.createDataFrame(data, ["city", "lat", "lng", "pm25"])
df_h3 = df.withColumn("h3_index", h3_udf(df.lat, df.lng))

print("Spark DataFrame with H3:\n")
df_h3.show(truncate=False)

print("✅ Spark + H3 integration working")

Spark DataFrame with H3:

+----------+-------+-------+-----+---------------+
|city      |lat    |lng    |pm25 |h3_index       |
+----------+-------+-------+-----+---------------+
|Delhi     |28.6139|77.209 |150.5|873da1146ffffff|
|Amritsar  |31.634 |74.8723|89.2 |87424d26dffffff|
|Chandigarh|30.7333|76.7794|112.8|873d14699ffffff|
+----------+-------+-------+-----+---------------+

✅ Spark + H3 integration working


## ✅ Setup Complete!

**Environment:**
- PySpark 3.5.0 + Delta Lake 3.1.0 ✓
- H3 spatial indexing (Res 7) ✓
- GeoPandas, PyDeck ✓
- Spark UI: http://localhost:4040

**Next: Learning Phase Notebooks**
1. Data ingestion (NASA FIRMS, OpenAQ, Open-Meteo)
2. H3 spatial binning patterns
3. PySpark batch processing
4. Delta Lake operations